# Clase 093 — LLE (Locally Linear Embedding)

Reducción de dimensionalidad **no lineal** por *manifold learning*: cada punto se reconstruye como combinación lineal de sus `k` vecinos y esa relación local se preserva en baja dimensión para "desenrollar" el swiss roll.

Requiere: `numpy`, `scikit-learn`, `scipy`, `matplotlib`.

## 1. Swiss roll de referencia

Manifold curvo 3D coloreado por la coordenada `t` a lo largo del rollo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll

np.random.seed(42)
X, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)
print("swiss roll:", X.shape, "| t:", t.shape)

fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=t, cmap="Spectral", s=8)
ax.set_title("Swiss roll 3D (color = t)")
plt.tight_layout(); plt.show()

## 2. LLE estándar desenrolla el rollo

`LocallyLinearEmbedding` con `n_neighbors=10`. Si el embedding es bueno, una coordenada debe ordenarse según `t`.

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding
from scipy.stats import spearmanr

lle = LocallyLinearEmbedding(n_neighbors=10, n_components=2,
                             eigen_solver="dense", random_state=42)
Z = lle.fit_transform(X)
print("embedding:", Z.shape, "| error reconstruccion:", round(lle.reconstruction_error_, 6))

rho = max(abs(spearmanr(Z[:, 0], t).statistic),
          abs(spearmanr(Z[:, 1], t).statistic))
print(f"|Spearman| mejor coordenada vs t = {rho:.4f}")
assert rho > 0.90, "LLE deberia preservar el orden a lo largo del rollo"

plt.figure(figsize=(7, 5))
plt.scatter(Z[:, 0], Z[:, 1], c=t, cmap="Spectral", s=8)
plt.xlabel("comp 1"); plt.ylabel("comp 2")
plt.title("LLE (n_neighbors=10): rollo desenrollado")
plt.tight_layout(); plt.show()

## 3. PCA aplasta donde LLE desenrolla

PCA es lineal: proyecta el rollo sin "abrirlo".

In [ ]:
from sklearn.decomposition import PCA

Zpca = PCA(n_components=2, random_state=42).fit_transform(X)
rho_pca = max(abs(spearmanr(Zpca[:, 0], t).statistic),
              abs(spearmanr(Zpca[:, 1], t).statistic))
print(f"|Spearman| PCA vs t = {rho_pca:.4f}  (peor que LLE)")

plt.figure(figsize=(7, 5))
plt.scatter(Zpca[:, 0], Zpca[:, 1], c=t, cmap="Spectral", s=8)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("PCA lineal: aplasta el rollo, mezcla los colores")
plt.tight_layout(); plt.show()

## 4. Barrido de `n_neighbors`

`k` chico desconecta el grafo; `k` grande pierde la estructura local y se parece a PCA.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, k in zip(axes.ravel(), (5, 10, 30, 100)):
    Zk = LocallyLinearEmbedding(n_neighbors=k, n_components=2,
                                eigen_solver="dense", random_state=42).fit_transform(X)
    ax.scatter(Zk[:, 0], Zk[:, 1], c=t, cmap="Spectral", s=6)
    ax.set_title(f"n_neighbors={k}")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Efecto de n_neighbors en LLE")
plt.tight_layout(); plt.show()

## 5. Modified LLE (MLLE)

`method='modified'` usa múltiples vectores de pesos por vecindario: embedding más estable cuando `n_neighbors > n_components`.

In [ ]:
mlle = LocallyLinearEmbedding(n_neighbors=12, n_components=2, method="modified",
                              eigen_solver="dense", random_state=42)
Zm = mlle.fit_transform(X)
rho_m = max(abs(spearmanr(Zm[:, 0], t).statistic),
            abs(spearmanr(Zm[:, 1], t).statistic))
print(f"|Spearman| MLLE vs t = {rho_m:.4f}")
assert Zm.shape == (1000, 2) and not np.isnan(Zm).any()

plt.figure(figsize=(7, 5))
plt.scatter(Zm[:, 0], Zm[:, 1], c=t, cmap="Spectral", s=8)
plt.xlabel("comp 1"); plt.ylabel("comp 2")
plt.title("MLLE (n_neighbors=12): embedding mas limpio")
plt.tight_layout(); plt.show()

## 6. LLE sobre datos reales (`load_digits`)

Proyección 2D coloreada por dígito: algunas clases se separan.

In [ ]:
from sklearn.datasets import load_digits

Xd, yd = load_digits(return_X_y=True)
Zd = LocallyLinearEmbedding(n_neighbors=15, n_components=2,
                            eigen_solver="dense", random_state=42).fit_transform(Xd)
plt.figure(figsize=(7, 5))
sc = plt.scatter(Zd[:, 0], Zd[:, 1], c=yd, cmap="tab10", s=10, alpha=0.7)
plt.colorbar(sc, label="digito")
plt.title("LLE sobre digits (64D -> 2D)")
plt.tight_layout(); plt.show()
print("embedding digits:", Zd.shape)

## Ejercicios

1. **Swiss roll básico:** generá 1000 puntos, aplicá `LLE(n_neighbors=10, n_components=2)` y coloreá por `t`. Verificá que quede desenrollado.
2. **Comparación con PCA:** discutí por qué PCA aplasta el rollo (sección 3).
3. **Barrido de `n_neighbors`:** probá `{5, 10, 30, 100}` en una grilla 2x2 y describí los extremos.
4. **Modified LLE:** repetí con `method='modified'` y compará limpieza del embedding.

## Conclusiones

- LLE preserva relaciones lineales **locales**: desenrolla manifolds curvos donde PCA falla.
- `n_neighbors` es el hiperparámetro clave: muy chico desconecta, muy grande imita a PCA.
- MLLE (`method='modified'`) es más estable cuando `n_neighbors > n_components`.
- LLE no preserva distancias globales (para eso, Isomap) ni tiene un `transform` barato para datos nuevos.

## ✅ Soluciones de los ejercicios

Cinco ejercicios de Locally Linear Embedding sobre Swiss roll y `load_digits`. `n_jobs=1`.

**Ejercicio 1 — Swiss roll básico.** LLE `n_neighbors=10, n_components=2`; coloreamos por la coordenada original `t` para ver el rollo desenrollado.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll, load_digits
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.decomposition import PCA

X, t = make_swiss_roll(n_samples=1000, random_state=42)
lle = LocallyLinearEmbedding(n_neighbors=10, n_components=2, random_state=42)
Z = lle.fit_transform(X)
plt.figure(figsize=(6, 4))
plt.scatter(Z[:, 0], Z[:, 1], c=t, cmap='viridis', s=8)
plt.title('LLE desenrolla el swiss roll'); plt.tight_layout(); plt.show()
print('reconstruction_error_:', round(float(lle.reconstruction_error_), 6))

**Ejercicio 2 — Comparación con PCA.** PCA es lineal: aplasta el rollo en vez de desenrollarlo.

In [ ]:
Zpca = PCA(n_components=2).fit_transform(X)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(Z[:, 0], Z[:, 1], c=t, cmap='viridis', s=8); ax[0].set_title('LLE (desenrolla)')
ax[1].scatter(Zpca[:, 0], Zpca[:, 1], c=t, cmap='viridis', s=8); ax[1].set_title('PCA (aplasta)')
plt.tight_layout(); plt.show()
print('PCA proyecta sobre un plano: mezcla puntos lejanos en el manifold.')

**Ejercicio 3 — Barrido de `n_neighbors`.** Pocos vecinos = fragmenta; demasiados = pierde la no-linealidad (se parece a PCA).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for ax, k in zip(axes.ravel(), [5, 10, 30, 100]):
    Zk = LocallyLinearEmbedding(n_neighbors=k, n_components=2,
                                random_state=42).fit_transform(X)
    ax.scatter(Zk[:, 0], Zk[:, 1], c=t, cmap='viridis', s=6)
    ax.set_title(f'n_neighbors={k}')
plt.tight_layout(); plt.show()
print('k chico: parches inconexos | k grande: se vuelve casi lineal.')

**Ejercicio 4 — Modified LLE.** `method='modified'` usa múltiples pesos por vecindad y suele dar un embedding más limpio.

In [ ]:
Zmod = LocallyLinearEmbedding(n_neighbors=12, n_components=2, method='modified',
                              random_state=42).fit_transform(X)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(Z[:, 0], Z[:, 1], c=t, cmap='viridis', s=8); ax[0].set_title('LLE estandar')
ax[1].scatter(Zmod[:, 0], Zmod[:, 1], c=t, cmap='viridis', s=8); ax[1].set_title('Modified LLE')
plt.tight_layout(); plt.show()
print('Modified LLE reduce la distorsion cuando hay mas vecinos que dimensiones.')

**Ejercicio 5 — LLE sobre datos reales.** `load_digits` (64D) a 2D, coloreado por dígito.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
sel = np.arange(1000)  # submuestra para runtime
Zd = LocallyLinearEmbedding(n_neighbors=15, n_components=2,
                            random_state=42).fit_transform(Xd[sel])
plt.figure(figsize=(6, 5))
sc = plt.scatter(Zd[:, 0], Zd[:, 1], c=yd[sel], cmap='tab10', s=10)
plt.colorbar(sc, label='digito'); plt.title('LLE sobre digits (2D)')
plt.tight_layout(); plt.show()
print('Algunos digitos se separan en islas; otros se solapan (LLE es no supervisado).')